# Set 05 – Logistische Regression mit scikit-learn

Die logistische Regression sagt keine kontinuierliche Zahl voraus, sondern schätzt eine Wahrscheinlichkeit für eine Klasse. Wir erzeugen binäre Daten, bauen eine Pipeline und untersuchen Schwellenwert, Recall, Precision, F1, Konfusionsmatrix und ROC-Kurve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. Kontrollierte binäre Daten

Klasse 1 ist absichtlich seltener. Das macht sichtbar, warum Accuracy allein nicht ausreicht.

In [ ]:
X_array, y_array = make_classification(
    n_samples=600,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.72, 0.28],
    class_sep=1.25,
    flip_y=0.04,
    random_state=42,
)
X = pd.DataFrame(X_array, columns=["messwert_1", "messwert_2"])
y = pd.Series(y_array, name="klasse")

print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

## 2. Klassen visualisieren

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for klasse, farbe in [(0, "#4C78A8"), (1, "#E45756")]:
    maske = y == klasse
    ax.scatter(
        X.loc[maske, "messwert_1"], X.loc[maske, "messwert_2"],
        label=f"Klasse {klasse}", alpha=0.7, color=farbe
    )
ax.set_xlabel("Messwert 1")
ax.set_ylabel("Messwert 2")
ax.set_title("Kontrollierte Klassifikationsdaten")
ax.legend()
plt.show()

## 3. Stratifiziert aufteilen

stratify=y erhält die Klassenanteile ungefähr in Trainings- und Testdaten.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Training:")
print(y_train.value_counts(normalize=True).round(3))
print("Test:")
print(y_test.value_counts(normalize=True).round(3))

## 4. fit und transform beim Scaler

StandardScaler.fit_transform() lernt Mittelwerte und Skalierungen aus den Trainingsdaten und transformiert sie. Für Testdaten wird ausschließlich transform() verwendet.

In [ ]:
scaler_demo = StandardScaler()
X_train_skaliert = scaler_demo.fit_transform(X_train)
X_test_skaliert = scaler_demo.transform(X_test)

print("Gelernte Mittelwerte:", scaler_demo.mean_.round(3))
print("Mittelwerte der skalierten Trainingsspalten:", X_train_skaliert.mean(axis=0).round(3))

## 5. Pipeline aus Skalierung und Modell

Beim Aufruf von pipeline.fit() führt scikit-learn intern fit_transform() für den Scaler und fit() für LogisticRegression aus. predict() und predict_proba() transformieren neue Daten automatisch mit dem bereits gelernten Scaler.

In [ ]:
pipeline = Pipeline([
    ("skalierung", StandardScaler()),
    ("modell", LogisticRegression()),
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

## 6. Wahrscheinlichkeiten und Klassen

predict_proba() liefert Wahrscheinlichkeiten. predict() verwendet bei binärer LogisticRegression standardmäßig den Schwellenwert 0,5.

In [ ]:
ergebnisse = X_test.copy()
ergebnisse["tatsaechlich"] = y_test
ergebnisse["wahrscheinlichkeit_klasse_1"] = y_proba
ergebnisse["vorhersage_bei_0_5"] = y_pred
display(ergebnisse.sort_values("wahrscheinlichkeit_klasse_1", ascending=False).head(10).round(3))

## 7. Klassifikationsmetriken

- Accuracy: Anteil aller richtigen Entscheidungen.
- Precision: Wie viele positive Vorhersagen waren tatsächlich positiv?
- Recall: Wie viele tatsächlich positive Fälle wurden gefunden?
- F1: harmonischer Mittelwert aus Precision und Recall.

Welcher Wert wichtig ist, hängt vom Anwendungsfall ab.

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}")

bericht = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T
display(bericht.round(3))

## 8. Konfusionsmatrix

Die Matrix trennt richtige und falsche Entscheidungen nach tatsächlicher Klasse. Falsch-negative Fälle sind tatsächlich Klasse 1, wurden aber als 0 vorhergesagt.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Klasse 0", "Klasse 1"], cmap="Blues"
)
plt.title("Konfusionsmatrix bei Schwellenwert 0,5")
plt.show()

## 9. Schwellenwert und Recall

Ein niedrigerer Schwellenwert erkennt meist mehr positive Fälle und erhöht damit den Recall. Gleichzeitig können mehr Fehlalarme entstehen und die Precision sinken.

In [ ]:
zeilen = []
for schwelle in [0.30, 0.50, 0.70]:
    entscheidung = (y_proba >= schwelle).astype(int)
    zeilen.append({
        "Schwellenwert": schwelle,
        "Accuracy": accuracy_score(y_test, entscheidung),
        "Precision": precision_score(y_test, entscheidung),
        "Recall": recall_score(y_test, entscheidung),
        "F1": f1_score(y_test, entscheidung),
        "positive Vorhersagen": int(entscheidung.sum()),
    })

display(pd.DataFrame(zeilen).round(3))

## 10. ROC-Kurve

Die ROC-Kurve zeigt Recall gegen Fehlalarmrate für viele mögliche Schwellenwerte. Die Fläche darunter ist die ROC-AUC.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Zufall")
plt.title("ROC-Kurve")
plt.legend()
plt.show()

## 11. Wahrscheinlichkeitsfläche

Die Farbe zeigt die geschätzte Wahrscheinlichkeit für Klasse 1. Die schwarze Linie markiert ungefähr die Entscheidungsgrenze bei 0,5.

In [ ]:
x1 = np.linspace(X["messwert_1"].min() - 0.5, X["messwert_1"].max() + 0.5, 220)
x2 = np.linspace(X["messwert_2"].min() - 0.5, X["messwert_2"].max() + 0.5, 220)
gitter_1, gitter_2 = np.meshgrid(x1, x2)
gitter = pd.DataFrame({
    "messwert_1": gitter_1.ravel(),
    "messwert_2": gitter_2.ravel(),
})
gitter_proba = pipeline.predict_proba(gitter)[:, 1].reshape(gitter_1.shape)

fig, ax = plt.subplots(figsize=(9, 6))
flaeche = ax.contourf(gitter_1, gitter_2, gitter_proba, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.65)
ax.contour(gitter_1, gitter_2, gitter_proba, levels=[0.5], colors="black", linewidths=2)
ax.scatter(X_test["messwert_1"], X_test["messwert_2"], c=y_test, cmap="bwr", edgecolor="white", s=35)
ax.set_xlabel("Messwert 1")
ax.set_ylabel("Messwert 2")
ax.set_title("Geschätzte Wahrscheinlichkeit und Entscheidungsgrenze")
fig.colorbar(flaeche, ax=ax, label="P(Klasse 1)")
plt.show()